# 01 - Single Izhikevich Neuron

## Objetivo
Implementar y visualizar una neurona Izhikevich individual.

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, IntSlider
import ipywidgets as widgets

# -----------------------------
# 1. Generadores de input
# -----------------------------

def generate_input(
    input_type="constant",
    t=None,
    amplitude=10.0,
    start=100.0,
    end=900.0,
    frequency=5.0,
    noise_level=2.0
):
    """
    Genera diferentes tipos de input I(t).

    input_type:
    - constant: corriente constante entre start y end
    - pulse: pulso breve
    - ramp: rampa creciente
    - sinusoidal: señal sinusoidal
    - noisy: corriente constante + ruido
    - motor_plan: comando motor artificial tipo subida-mantenimiento-bajada
    """

    I = np.zeros_like(t)

    active = (t >= start) & (t <= end)

    if input_type == "constant":
        I[active] = amplitude

    elif input_type == "pulse":
        pulse_end = start + 100
        active_pulse = (t >= start) & (t <= pulse_end)
        I[active_pulse] = amplitude

    elif input_type == "ramp":
        duration = end - start
        I[active] = amplitude * (t[active] - start) / duration

    elif input_type == "sinusoidal":
        I[active] = amplitude * (0.5 + 0.5 * np.sin(2 * np.pi * frequency * (t[active] - start) / 1000))

    elif input_type == "noisy":
        I[active] = amplitude + noise_level * np.random.randn(np.sum(active))

    elif input_type == "motor_plan":
        # Comando motor artificial:
        # fase 1: reposo
        # fase 2: aumento progresivo
        # fase 3: contracción sostenida
        # fase 4: relajación progresiva

        rise_start = start
        rise_end = start + 200

        hold_start = rise_end
        hold_end = end - 200

        fall_start = hold_end
        fall_end = end

        rise = (t >= rise_start) & (t < rise_end)
        hold = (t >= hold_start) & (t < hold_end)
        fall = (t >= fall_start) & (t <= fall_end)

        I[rise] = amplitude * (t[rise] - rise_start) / (rise_end - rise_start)
        I[hold] = amplitude
        I[fall] = amplitude * (1 - (t[fall] - fall_start) / (fall_end - fall_start))

    return I


# -----------------------------
# 2. Simulación Izhikevich
# -----------------------------

def simulate_izhikevich(
    a=0.02,
    b=0.2,
    c=-65.0,
    d=8.0,
    input_type="constant",
    amplitude=10.0,
    total_time=1000,
    dt=0.5,
    input_start=100,
    input_end=900,
    frequency=5.0,
    noise_level=2.0
):
    """
    Simula una neurona de Izhikevich usando método de Euler.
    """

    t = np.arange(0, total_time, dt)
    n = len(t)

    v = np.zeros(n)
    u = np.zeros(n)
    spikes = np.zeros(n)

    # Condiciones iniciales típicas
    v[0] = -65.0
    u[0] = b * v[0]

    I = generate_input(
        input_type=input_type,
        t=t,
        amplitude=amplitude,
        start=input_start,
        end=input_end,
        frequency=frequency,
        noise_level=noise_level
    )

    for i in range(1, n):
        dv = 0.04 * v[i-1]**2 + 5 * v[i-1] + 140 - u[i-1] + I[i-1]
        du = a * (b * v[i-1] - u[i-1])

        v[i] = v[i-1] + dt * dv
        u[i] = u[i-1] + dt * du

        if v[i] >= 30:
            v[i-1] = 30      # Para graficar el spike
            v[i] = c         # Reset de v
            u[i] = u[i] + d  # Reset de u
            spikes[i] = 1

    return t, v, u, I, spikes


# -----------------------------
# 3. Función de visualización
# -----------------------------

def plot_simulation(
    neuron_type="Regular Spiking",
    input_type="constant",
    amplitude=10.0,
    a=0.02,
    b=0.2,
    c=-65.0,
    d=8.0,
    total_time=1000,
    dt=0.5,
    input_start=100,
    input_end=900,
    frequency=5.0,
    noise_level=2.0
):
    """
    Corre la simulación y grafica los resultados.
    """

    # Parámetros predefinidos por tipo neuronal
    if neuron_type == "Regular Spiking":
        a, b, c, d = 0.02, 0.2, -65, 8

    elif neuron_type == "Intrinsically Bursting":
        a, b, c, d = 0.02, 0.2, -55, 4

    elif neuron_type == "Chattering":
        a, b, c, d = 0.02, 0.2, -50, 2

    elif neuron_type == "Fast Spiking":
        a, b, c, d = 0.1, 0.2, -65, 2

    elif neuron_type == "Low Threshold Spiking":
        a, b, c, d = 0.02, 0.25, -65, 2

    elif neuron_type == "Custom":
        pass

    t, v, u, I, spikes = simulate_izhikevich(
        a=a,
        b=b,
        c=c,
        d=d,
        input_type=input_type,
        amplitude=amplitude,
        total_time=total_time,
        dt=dt,
        input_start=input_start,
        input_end=input_end,
        frequency=frequency,
        noise_level=noise_level
    )

    spike_times = t[spikes == 1]
    firing_rate = len(spike_times) / (total_time / 1000)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    axes[0].plot(t, v)
    axes[0].set_ylabel("v(t)")
    axes[0].set_title(
        f"Izhikevich neuron - {neuron_type} | Spikes: {len(spike_times)} | Firing rate: {firing_rate:.2f} Hz"
    )
    axes[0].grid(True)

    axes[1].plot(t, I)
    axes[1].set_ylabel("Input I(t)")
    axes[1].grid(True)

    axes[2].eventplot(spike_times, lineoffsets=1, linelengths=0.8)
    axes[2].set_ylabel("Spikes")
    axes[2].set_xlabel("Time (ms)")
    axes[2].set_yticks([])
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()


# -----------------------------
# 4. GUI interactiva
# -----------------------------

interact(
    plot_simulation,

    neuron_type=Dropdown(
        options=[
            "Regular Spiking",
            "Intrinsically Bursting",
            "Chattering",
            "Fast Spiking",
            "Low Threshold Spiking",
            "Custom"
        ],
        value="Regular Spiking",
        description="Neuron"
    ),

    input_type=Dropdown(
        options=[
            "constant",
            "pulse",
            "ramp",
            "sinusoidal",
            "noisy",
            "motor_plan"
        ],
        value="constant",
        description="Input"
    ),

    amplitude=FloatSlider(
        value=10.0,
        min=0.0,
        max=30.0,
        step=0.5,
        description="Amplitude"
    ),

    a=FloatSlider(
        value=0.02,
        min=0.001,
        max=0.2,
        step=0.001,
        description="a"
    ),

    b=FloatSlider(
        value=0.2,
        min=0.01,
        max=0.4,
        step=0.01,
        description="b"
    ),

    c=FloatSlider(
        value=-65.0,
        min=-90.0,
        max=-40.0,
        step=1.0,
        description="c"
    ),

    d=FloatSlider(
        value=8.0,
        min=0.0,
        max=20.0,
        step=0.5,
        description="d"
    ),

    total_time=IntSlider(
        value=1000,
        min=200,
        max=3000,
        step=100,
        description="Time"
    ),

    dt=FloatSlider(
        value=0.5,
        min=0.1,
        max=2.0,
        step=0.1,
        description="dt"
    ),

    input_start=IntSlider(
        value=100,
        min=0,
        max=1000,
        step=50,
        description="I start"
    ),

    input_end=IntSlider(
        value=900,
        min=100,
        max=3000,
        step=50,
        description="I end"
    ),

    frequency=FloatSlider(
        value=5.0,
        min=0.5,
        max=30.0,
        step=0.5,
        description="Freq"
    ),

    noise_level=FloatSlider(
        value=2.0,
        min=0.0,
        max=10.0,
        step=0.5,
        description="Noise"
    )
);

interactive(children=(Dropdown(description='Neuron', options=('Regular Spiking', 'Intrinsically Bursting', 'Ch…

# 02 - Pull of Izhikevich Neurons

## Objetivo
Implementar y visualizar un grupo de neuronas Izhikevich

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

## Intput generator

In [18]:
def generate_input(
    input_type="motor_plan",
    t=None,
    amplitude=10.0,
    start=100.0,
    end=900.0,
    frequency=5.0
):
    """
    Genera un input común I(t) para todas las neuronas.

    input_type:
    - constant: input constante entre start y end
    - pulse: pulso breve
    - ramp: rampa creciente
    - sinusoidal: input oscilatorio
    - motor_plan: subida, mantenimiento y bajada
    """

    I = np.zeros_like(t)

    active = (t >= start) & (t <= end)

    if input_type == "constant":
        I[active] = amplitude

    elif input_type == "pulse":
        pulse_end = start + 100
        active_pulse = (t >= start) & (t <= pulse_end)
        I[active_pulse] = amplitude

    elif input_type == "ramp":
        duration = end - start
        I[active] = amplitude * (t[active] - start) / duration

    elif input_type == "sinusoidal":
        I[active] = amplitude * (
            0.5 + 0.5 * np.sin(2 * np.pi * frequency * (t[active] - start) / 1000)
        )

    elif input_type == "motor_plan":
        # Plan motor artificial:
        # reposo -> subida -> contracción sostenida -> relajación

        rise_start = start
        rise_end = start + 200

        hold_start = rise_end
        hold_end = end - 200

        fall_start = hold_end
        fall_end = end

        rise = (t >= rise_start) & (t < rise_end)
        hold = (t >= hold_start) & (t < hold_end)
        fall = (t >= fall_start) & (t <= fall_end)

        I[rise] = amplitude * (t[rise] - rise_start) / (rise_end - rise_start)
        I[hold] = amplitude
        I[fall] = amplitude * (1 - (t[fall] - fall_start) / (fall_end - fall_start))

    return I

## Pool generator

In [19]:
def simulate_neuron_pool(
    n_neurons=30,
    total_time=1000,
    dt=0.5,
    input_type="motor_plan",
    amplitude=10.0,
    input_start=100,
    input_end=900,
    frequency=5.0,
    base_a=0.02,
    base_b=0.2,
    base_c=-65.0,
    base_d=8.0,
    parameter_noise=0.05,
    input_noise=1.0,
    random_seed=42
):
    """
    Simula un pool de neuronas Izhikevich independientes.

    Todas reciben el mismo input común, pero:
    - cada neurona tiene pequeñas variaciones en a, b, c, d
    - cada neurona recibe ruido individual en el input

    Retorna:
    - t: vector de tiempo
    - V: matriz voltaje [n_neurons, time]
    - U: matriz recuperación [n_neurons, time]
    - spikes: matriz binaria [n_neurons, time]
    - I_common: input común
    - I_all: input total por neurona
    - params: parámetros individuales
    """

    rng = np.random.default_rng(random_seed)

    t = np.arange(0, total_time, dt)
    n_steps = len(t)

    # Input común para todas las neuronas
    I_common = generate_input(
        input_type=input_type,
        t=t,
        amplitude=amplitude,
        start=input_start,
        end=input_end,
        frequency=frequency
    )

    # -----------------------------
    # Parámetros individuales
    # -----------------------------
    # La idea es que todas las neuronas sean parecidas,
    # pero no idénticas.

    a = base_a * (1 + parameter_noise * rng.normal(size=n_neurons))
    b = base_b * (1 + parameter_noise * rng.normal(size=n_neurons))
    c = base_c + abs(base_c) * parameter_noise * rng.normal(size=n_neurons)
    d = base_d * (1 + parameter_noise * rng.normal(size=n_neurons))

    # Evitamos valores absurdos por ruido
    a = np.clip(a, 0.001, 0.2)
    b = np.clip(b, 0.01, 0.4)
    c = np.clip(c, -90, -40)
    d = np.clip(d, 0.1, 20)

    params = {
        "a": a,
        "b": b,
        "c": c,
        "d": d
    }

    # -----------------------------
    # Matrices de simulación
    # -----------------------------

    V = np.zeros((n_neurons, n_steps))
    U = np.zeros((n_neurons, n_steps))
    spikes = np.zeros((n_neurons, n_steps))

    # Condiciones iniciales
    V[:, 0] = -65.0
    U[:, 0] = b * V[:, 0]

    # Input total por neurona
    # Todas tienen el input común + ruido individual
    I_all = np.tile(I_common, (n_neurons, 1))
    I_all += input_noise * rng.normal(size=(n_neurons, n_steps))

    # -----------------------------
    # Simulación temporal
    # -----------------------------

    for i in range(1, n_steps):

        dv = 0.04 * V[:, i-1]**2 + 5 * V[:, i-1] + 140 - U[:, i-1] + I_all[:, i-1]
        du = a * (b * V[:, i-1] - U[:, i-1])

        V[:, i] = V[:, i-1] + dt * dv
        U[:, i] = U[:, i-1] + dt * du

        fired = V[:, i] >= 30

        # Para que el spike se vea en el gráfico
        V[fired, i-1] = 30

        # Reset después del spike
        V[fired, i] = c[fired]
        U[fired, i] = U[fired, i] + d[fired]

        spikes[fired, i] = 1

    return t, V, U, spikes, I_common, I_all, params

## Activity record

In [20]:
def plot_neuron_pool_results(
    t,
    V,
    spikes,
    I_common,
    dt=0.5,
    n_voltage_traces=5,
    bin_size_ms=20
):
    """
    Grafica resultados del pool neuronal.
    """

    n_neurons, n_steps = V.shape

    # Actividad poblacional
    time_bins, activity = compute_population_activity(
        spikes,
        dt=dt,
        bin_size_ms=bin_size_ms
    )

    fig, axes = plt.subplots(5, 1, figsize=(15, 16), sharex=False)

    # -----------------------------
    # 1. Input común
    # -----------------------------
    axes[0].plot(t, I_common)
    axes[0].set_title("Input común recibido por el pool neuronal")
    axes[0].set_ylabel("I(t)")
    axes[0].grid(True)

    # -----------------------------
    # 2. Voltaje de algunas neuronas
    # -----------------------------
    selected_neurons = np.linspace(
        0,
        n_neurons - 1,
        min(n_voltage_traces, n_neurons),
        dtype=int
    )

    for neuron_idx in selected_neurons:
        axes[1].plot(t, V[neuron_idx], label=f"Neuron {neuron_idx}")

    axes[1].set_title("Potencial de membrana de algunas neuronas")
    axes[1].set_ylabel("v(t)")
    axes[1].legend(loc="upper right")
    axes[1].grid(True)

    # -----------------------------
    # 3. Raster plot
    # -----------------------------
    for neuron_idx in range(n_neurons):
        spike_times = t[spikes[neuron_idx] == 1]
        axes[2].vlines(
            spike_times,
            neuron_idx + 0.5,
            neuron_idx + 1.5,
            linewidth=0.8
        )

    axes[2].set_title("Raster plot: spikes por neurona")
    axes[2].set_ylabel("Neurona")
    axes[2].set_ylim(0.5, n_neurons + 0.5)
    axes[2].grid(True)

    # -----------------------------
    # 4. Mapa de calor del voltaje
    # -----------------------------
    im = axes[3].imshow(
        V,
        aspect="auto",
        origin="lower",
        extent=[t[0], t[-1], 0, n_neurons],
        interpolation="nearest"
    )

    axes[3].set_title("Mapa de calor del potencial de membrana")
    axes[3].set_ylabel("Neurona")
    axes[3].set_xlabel("Tiempo (ms)")
    fig.colorbar(im, ax=axes[3], label="v(t)")

    # -----------------------------
    # 5. Actividad poblacional
    # -----------------------------
    axes[4].bar(
        time_bins,
        activity,
        width=bin_size_ms,
        align="edge"
    )

    axes[4].set_title(f"Actividad poblacional: spikes cada {bin_size_ms} ms")
    axes[4].set_ylabel("Spikes")
    axes[4].set_xlabel("Tiempo (ms)")
    axes[4].grid(True)

    plt.tight_layout()
    plt.show()

In [21]:
def compute_population_activity(spikes, dt=0.5, bin_size_ms=20):
    """
    Calcula actividad poblacional agrupando spikes en ventanas temporales.

    spikes:
    matriz [n_neurons, time]

    Retorna:
    - activity: spikes por ventana
    """

    n_neurons, n_steps = spikes.shape

    steps_per_bin = int(bin_size_ms / dt)

    n_bins = n_steps // steps_per_bin

    activity = np.zeros(n_bins)
    time_bins = np.zeros(n_bins)

    for i in range(n_bins):
        start = i * steps_per_bin
        end = start + steps_per_bin

        activity[i] = spikes[:, start:end].sum()
        time_bins[i] = start * dt

    return time_bins, activity

In [22]:
def run_pool_experiment(
    n_neurons=30,
    input_type="motor_plan",
    amplitude=10.0,
    total_time=1000,
    dt=0.5,
    input_start=100,
    input_end=900,
    frequency=5.0,
    parameter_noise=0.05,
    input_noise=1.0,
    random_seed=42,
    bin_size_ms=20
):
    """
    Corre una simulación completa del pool neuronal y grafica resultados.
    """

    t, V, U, spikes, I_common, I_all, params = simulate_neuron_pool(
        n_neurons=n_neurons,
        total_time=total_time,
        dt=dt,
        input_type=input_type,
        amplitude=amplitude,
        input_start=input_start,
        input_end=input_end,
        frequency=frequency,
        parameter_noise=parameter_noise,
        input_noise=input_noise,
        random_seed=random_seed
    )

    total_spikes = spikes.sum()
    firing_rate_mean = total_spikes / n_neurons / (total_time / 1000)

    print("Resumen de simulación")
    print("---------------------")
    print(f"Número de neuronas: {n_neurons}")
    print(f"Spikes totales: {int(total_spikes)}")
    print(f"Firing rate medio: {firing_rate_mean:.2f} Hz")
    print(f"Ruido en parámetros: {parameter_noise}")
    print(f"Ruido en input: {input_noise}")

    plot_neuron_pool_results(
        t=t,
        V=V,
        spikes=spikes,
        I_common=I_common,
        dt=dt,
        bin_size_ms=bin_size_ms
    )

    return t, V, U, spikes, I_common, I_all, params

In [23]:
interact(
    run_pool_experiment,

    n_neurons=IntSlider(
        value=30,
        min=5,
        max=100,
        step=5,
        description="Neurons"
    ),

    input_type=Dropdown(
        options=[
            "constant",
            "pulse",
            "ramp",
            "sinusoidal",
            "motor_plan"
        ],
        value="motor_plan",
        description="Input"
    ),

    amplitude=FloatSlider(
        value=10.0,
        min=0.0,
        max=30.0,
        step=0.5,
        description="Amplitude"
    ),

    total_time=IntSlider(
        value=1000,
        min=500,
        max=3000,
        step=100,
        description="Time"
    ),

    dt=FloatSlider(
        value=0.5,
        min=0.1,
        max=2.0,
        step=0.1,
        description="dt"
    ),

    input_start=IntSlider(
        value=100,
        min=0,
        max=2000,
        step=50,
        description="I start"
    ),

    input_end=IntSlider(
        value=900,
        min=100,
        max=3000,
        step=50,
        description="I end"
    ),

    frequency=FloatSlider(
        value=5.0,
        min=0.5,
        max=30.0,
        step=0.5,
        description="Freq"
    ),

    parameter_noise=FloatSlider(
        value=0.05,
        min=0.0,
        max=0.3,
        step=0.01,
        description="Param noise"
    ),

    input_noise=FloatSlider(
        value=1.0,
        min=0.0,
        max=10.0,
        step=0.5,
        description="Input noise"
    ),

    random_seed=IntSlider(
        value=42,
        min=1,
        max=100,
        step=1,
        description="Seed"
    ),

    bin_size_ms=IntSlider(
        value=20,
        min=5,
        max=100,
        step=5,
        description="Bin ms"
    )
);

interactive(children=(IntSlider(value=30, description='Neurons', min=5, step=5), Dropdown(description='Input',…

In [24]:
!pip install numpy matplotlib ipywidgets networkx

In [25]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from ipywidgets import interact, IntSlider, FloatSlider, Dropdown, Checkbox

In [26]:
# ============================================================
# 1. GENERADOR DE INPUTS
# ============================================================

def generate_input(
    input_type="motor_plan",
    t=None,
    amplitude=10.0,
    start=100.0,
    end=900.0,
    frequency=5.0
):
    """
    Genera un input común I(t) para todas las neuronas.

    Tipos disponibles:
    - constant: corriente constante
    - pulse: pulso breve
    - ramp: rampa creciente
    - sinusoidal: input oscilatorio
    - motor_plan: subida, mantenimiento y bajada
    """

    I = np.zeros_like(t)
    active = (t >= start) & (t <= end)

    if input_type == "constant":
        I[active] = amplitude

    elif input_type == "pulse":
        pulse_end = start + 100
        active_pulse = (t >= start) & (t <= pulse_end)
        I[active_pulse] = amplitude

    elif input_type == "ramp":
        duration = max(end - start, 1)
        I[active] = amplitude * (t[active] - start) / duration

    elif input_type == "sinusoidal":
        I[active] = amplitude * (
            0.5 + 0.5 * np.sin(2 * np.pi * frequency * (t[active] - start) / 1000)
        )

    elif input_type == "motor_plan":
        # Plan motor artificial:
        # reposo -> subida -> contracción sostenida -> relajación

        rise_start = start
        rise_end = start + 200

        hold_start = rise_end
        hold_end = end - 200

        fall_start = hold_end
        fall_end = end

        rise = (t >= rise_start) & (t < rise_end)
        hold = (t >= hold_start) & (t < hold_end)
        fall = (t >= fall_start) & (t <= fall_end)

        if rise_end > rise_start:
            I[rise] = amplitude * (t[rise] - rise_start) / (rise_end - rise_start)

        I[hold] = amplitude

        if fall_end > fall_start:
            I[fall] = amplitude * (1 - (t[fall] - fall_start) / (fall_end - fall_start))

    return I

In [27]:
# ============================================================
# 2. SIMULACIÓN DEL POOL DE NEURONAS IZHIKEVICH
# ============================================================

def simulate_neuron_pool(
    n_neurons=30,
    total_time=1000,
    dt=0.5,
    input_type="motor_plan",
    amplitude=10.0,
    input_start=100,
    input_end=900,
    frequency=5.0,
    base_a=0.02,
    base_b=0.2,
    base_c=-65.0,
    base_d=8.0,
    parameter_noise=0.05,
    input_noise=1.0,
    random_seed=42
):
    """
    Simula un pool de neuronas Izhikevich independientes.

    Todas reciben el mismo input común, pero:
    - cada neurona tiene pequeñas variaciones en sus parámetros;
    - cada neurona recibe ruido individual en el input.

    Todavía NO hay conexiones entre neuronas.
    """

    rng = np.random.default_rng(random_seed)

    t = np.arange(0, total_time, dt)
    n_steps = len(t)

    # Input común
    I_common = generate_input(
        input_type=input_type,
        t=t,
        amplitude=amplitude,
        start=input_start,
        end=input_end,
        frequency=frequency
    )

    # --------------------------------------------------------
    # Parámetros individuales
    # --------------------------------------------------------
    # Todas las neuronas parten de los mismos parámetros base,
    # pero se les suma una pequeña variación aleatoria.

    a = base_a * (1 + parameter_noise * rng.normal(size=n_neurons))
    b = base_b * (1 + parameter_noise * rng.normal(size=n_neurons))
    c = base_c + abs(base_c) * parameter_noise * rng.normal(size=n_neurons)
    d = base_d * (1 + parameter_noise * rng.normal(size=n_neurons))

    # Evitamos valores extremos no deseados
    a = np.clip(a, 0.001, 0.2)
    b = np.clip(b, 0.01, 0.4)
    c = np.clip(c, -90, -40)
    d = np.clip(d, 0.1, 20)

    params = {
        "a": a,
        "b": b,
        "c": c,
        "d": d
    }

    # --------------------------------------------------------
    # Matrices de estado
    # --------------------------------------------------------

    V = np.zeros((n_neurons, n_steps))
    U = np.zeros((n_neurons, n_steps))
    spikes = np.zeros((n_neurons, n_steps))

    # Condiciones iniciales
    V[:, 0] = -65.0
    U[:, 0] = b * V[:, 0]

    # Input total por neurona:
    # input común + ruido individual
    I_all = np.tile(I_common, (n_neurons, 1))
    I_all += input_noise * rng.normal(size=(n_neurons, n_steps))

    # --------------------------------------------------------
    # Integración numérica por Euler
    # --------------------------------------------------------

    for i in range(1, n_steps):

        dv = 0.04 * V[:, i-1]**2 + 5 * V[:, i-1] + 140 - U[:, i-1] + I_all[:, i-1]
        du = a * (b * V[:, i-1] - U[:, i-1])

        V[:, i] = V[:, i-1] + dt * dv
        U[:, i] = U[:, i-1] + dt * du

        fired = V[:, i] >= 30

        # Marcamos el pico para que se vea en el gráfico
        V[fired, i-1] = 30

        # Reset de Izhikevich
        V[fired, i] = c[fired]
        U[fired, i] = U[fired, i] + d[fired]

        spikes[fired, i] = 1

    return t, V, U, spikes, I_common, I_all, params

In [28]:
# ============================================================
# 3. ACTIVIDAD POBLACIONAL
# ============================================================

def compute_population_activity(spikes, dt=0.5, bin_size_ms=20):
    """
    Cuenta cuántos spikes ocurren en ventanas de tiempo.
    """

    n_neurons, n_steps = spikes.shape

    steps_per_bin = max(int(bin_size_ms / dt), 1)
    n_bins = n_steps // steps_per_bin

    activity = np.zeros(n_bins)
    time_bins = np.zeros(n_bins)

    for i in range(n_bins):
        start = i * steps_per_bin
        end = start + steps_per_bin

        activity[i] = spikes[:, start:end].sum()
        time_bins[i] = start * dt

    return time_bins, activity

In [29]:
# ============================================================
# 4. DIAGRAMA DE ARQUITECTURA: INPUT -> POOL
# ============================================================

def plot_input_to_pool_grid(n_neurons=30, show_labels=True):
    """
    Visualiza la arquitectura actual:

    Input común -> pool de N neuronas.

    Este gráfico es solo una referencia visual.
    En esta etapa las neuronas NO están conectadas entre sí.
    """

    G = nx.DiGraph()

    input_node = "Input"
    G.add_node(input_node)

    neuron_nodes = [f"N{i+1}" for i in range(n_neurons)]

    for neuron in neuron_nodes:
        G.add_node(neuron)
        G.add_edge(input_node, neuron)

    # Posiciones
    pos = {}
    pos[input_node] = (0, 2)

    n_cols = int(np.ceil(np.sqrt(n_neurons)))
    n_rows = int(np.ceil(n_neurons / n_cols))

    spacing_x = 1.0
    spacing_y = 0.8

    for idx, neuron in enumerate(neuron_nodes):
        row = idx // n_cols
        col = idx % n_cols

        x = (col - n_cols / 2) * spacing_x
        y = -row * spacing_y

        pos[neuron] = (x, y)

    plt.figure(figsize=(12, 8))

    # Nodo input
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=[input_node],
        node_size=1800,
        node_color="lightgray",
        edgecolors="black"
    )

    # Nodos neuronales
    nx.draw_networkx_nodes(
        G,
        pos,
        nodelist=neuron_nodes,
        node_size=500 if n_neurons <= 50 else 250,
        node_color="white",
        edgecolors="black"
    )

    # Flechas
    nx.draw_networkx_edges(
        G,
        pos,
        arrows=True,
        arrowstyle="->",
        arrowsize=10,
        width=0.6,
        alpha=0.4
    )

    # Labels
    if show_labels:
        labels = {node: node for node in G.nodes()}
        font_size = 8 if n_neurons <= 50 else 5
    else:
        labels = {input_node: "Input"}
        font_size = 12

    nx.draw_networkx_labels(
        G,
        pos,
        labels=labels,
        font_size=font_size
    )

    plt.title(f"Arquitectura actual: input común conectado a {n_neurons} neuronas")
    plt.axis("off")
    plt.show()

In [30]:
# ============================================================
# 5. VISUALIZACIÓN DE RESULTADOS
# ============================================================

def plot_neuron_pool_results(
    t,
    V,
    spikes,
    I_common,
    dt=0.5,
    n_voltage_traces=5,
    bin_size_ms=20
):
    """
    Grafica:
    1. Input común.
    2. Potencial de membrana de algunas neuronas.
    3. Raster plot.
    4. Mapa de calor del voltaje.
    5. Actividad poblacional.
    """

    n_neurons, n_steps = V.shape

    time_bins, activity = compute_population_activity(
        spikes=spikes,
        dt=dt,
        bin_size_ms=bin_size_ms
    )

    fig, axes = plt.subplots(5, 1, figsize=(15, 18), sharex=False)

    # --------------------------------------------------------
    # 1. Input común
    # --------------------------------------------------------
    axes[0].plot(t, I_common)
    axes[0].set_title("Input común recibido por el pool neuronal")
    axes[0].set_ylabel("I(t)")
    axes[0].set_xlabel("Tiempo (ms)")
    axes[0].grid(True)

    # --------------------------------------------------------
    # 2. Voltaje de algunas neuronas
    # --------------------------------------------------------
    selected_neurons = np.linspace(
        0,
        n_neurons - 1,
        min(n_voltage_traces, n_neurons),
        dtype=int
    )

    for neuron_idx in selected_neurons:
        axes[1].plot(t, V[neuron_idx], label=f"Neuron {neuron_idx + 1}")

    axes[1].set_title("Potencial de membrana de algunas neuronas")
    axes[1].set_ylabel("v(t)")
    axes[1].set_xlabel("Tiempo (ms)")
    axes[1].legend(loc="upper right")
    axes[1].grid(True)

    # --------------------------------------------------------
    # 3. Raster plot
    # --------------------------------------------------------
    for neuron_idx in range(n_neurons):
        spike_times = t[spikes[neuron_idx] == 1]

        axes[2].vlines(
            spike_times,
            neuron_idx + 0.5,
            neuron_idx + 1.5,
            linewidth=0.8
        )

    axes[2].set_title("Raster plot: spikes por neurona")
    axes[2].set_ylabel("Neurona")
    axes[2].set_xlabel("Tiempo (ms)")
    axes[2].set_ylim(0.5, n_neurons + 0.5)
    axes[2].grid(True)

    # --------------------------------------------------------
    # 4. Mapa de calor del potencial de membrana
    # --------------------------------------------------------
    im = axes[3].imshow(
        V,
        aspect="auto",
        origin="lower",
        extent=[t[0], t[-1], 1, n_neurons],
        interpolation="nearest"
    )

    axes[3].set_title("Mapa de calor del potencial de membrana")
    axes[3].set_ylabel("Neurona")
    axes[3].set_xlabel("Tiempo (ms)")

    cbar = fig.colorbar(im, ax=axes[3])
    cbar.set_label("v(t)")

    # --------------------------------------------------------
    # 5. Actividad poblacional
    # --------------------------------------------------------
    axes[4].bar(
        time_bins,
        activity,
        width=bin_size_ms,
        align="edge"
    )

    axes[4].set_title(f"Actividad poblacional: spikes cada {bin_size_ms} ms")
    axes[4].set_ylabel("Cantidad de spikes")
    axes[4].set_xlabel("Tiempo (ms)")
    axes[4].grid(True)

    plt.tight_layout()
    plt.show()

In [31]:
# ============================================================
# 6. FUNCIÓN PRINCIPAL DE LA GUI
# ============================================================

def run_pool_gui(
    n_neurons=30,
    input_type="motor_plan",
    amplitude=10.0,
    total_time=1000,
    dt=0.5,
    input_start=100,
    input_end=900,
    frequency=5.0,
    parameter_noise=0.05,
    input_noise=1.0,
    random_seed=42,
    bin_size_ms=20,
    show_network_graph=True,
    show_network_labels=True
):
    """
    Función principal que corre toda la simulación y muestra gráficos.
    """

    # --------------------------------------------------------
    # Diagrama de arquitectura
    # --------------------------------------------------------
    if show_network_graph:
        plot_input_to_pool_grid(
            n_neurons=n_neurons,
            show_labels=show_network_labels and n_neurons <= 80
        )

    # --------------------------------------------------------
    # Simulación
    # --------------------------------------------------------
    t, V, U, spikes, I_common, I_all, params = simulate_neuron_pool(
        n_neurons=n_neurons,
        total_time=total_time,
        dt=dt,
        input_type=input_type,
        amplitude=amplitude,
        input_start=input_start,
        input_end=input_end,
        frequency=frequency,
        parameter_noise=parameter_noise,
        input_noise=input_noise,
        random_seed=random_seed
    )

    total_spikes = int(spikes.sum())
    duration_seconds = total_time / 1000
    firing_rate_mean = total_spikes / n_neurons / duration_seconds

    # --------------------------------------------------------
    # Resumen
    # --------------------------------------------------------
    print("Resumen de simulación")
    print("---------------------")
    print(f"Número de neuronas: {n_neurons}")
    print(f"Tipo de input: {input_type}")
    print(f"Amplitud del input: {amplitude}")
    print(f"Spikes totales: {total_spikes}")
    print(f"Firing rate medio: {firing_rate_mean:.2f} Hz")
    print(f"Ruido en parámetros: {parameter_noise}")
    print(f"Ruido en input: {input_noise}")
    print()
    print("Interpretación breve")
    print("--------------------")
    print("Todas las neuronas reciben el mismo input común.")
    print("Las diferencias entre neuronas aparecen por variación en parámetros y ruido individual.")
    print("En esta etapa todavía no hay conexiones sinápticas entre neuronas.")

    # --------------------------------------------------------
    # Resultados gráficos
    # --------------------------------------------------------
    plot_neuron_pool_results(
        t=t,
        V=V,
        spikes=spikes,
        I_common=I_common,
        dt=dt,
        bin_size_ms=bin_size_ms
    )

    return t, V, U, spikes, I_common, I_all, params

In [ ]:
# ============================================================
# 7. GUI INTERACTIVA
# ============================================================

interact(
    run_pool_gui,

    n_neurons=IntSlider(
        value=30,
        min=5,
        max=100,
        step=5,
        description="Neurons"
    ),

    input_type=Dropdown(
        options=[
            "constant",
            "pulse",
            "ramp",
            "sinusoidal",
            "motor_plan"
        ],
        value="motor_plan",
        description="Input"
    ),

    amplitude=FloatSlider(
        value=10.0,
        min=0.0,
        max=30.0,
        step=0.5,
        description="Amplitude"
    ),

    total_time=IntSlider(
        value=1000,
        min=500,
        max=3000,
        step=100,
        description="Time"
    ),

    dt=FloatSlider(
        value=0.5,
        min=0.1,
        max=2.0,
        step=0.1,
        description="dt"
    ),

    input_start=IntSlider(
        value=100,
        min=0,
        max=2000,
        step=50,
        description="I start"
    ),

    input_end=IntSlider(
        value=900,
        min=100,
        max=3000,
        step=50,
        description="I end"
    ),

    frequency=FloatSlider(
        value=5.0,
        min=0.5,
        max=30.0,
        step=0.5,
        description="Freq"
    ),

    parameter_noise=FloatSlider(
        value=0.05,
        min=0.0,
        max=0.3,
        step=0.01,
        description="Param noise"
    ),

    input_noise=FloatSlider(
        value=1.0,
        min=0.0,
        max=10.0,
        step=0.5,
        description="Input noise"
    ),

    random_seed=IntSlider(
        value=42,
        min=1,
        max=100,
        step=1,
        description="Seed"
    ),

    bin_size_ms=IntSlider(
        value=20,
        min=5,
        max=100,
        step=5,
        description="Bin ms"
    ),

    show_network_graph=Checkbox(
        value=True,
        description="Show graph"
    ),

    show_network_labels=Checkbox(
        value=True,
        description="Show labels"
    )
);

interactive(children=(IntSlider(value=30, description='Neurons', min=5, step=5), Dropdown(description='Input',…